# Laboratorio L14 - Verifiche difensive su un'applicazione con modello linguistico

B.4 - Cybersicurezza e IA

Il notebook contiene due controlli che uno sviluppatore o un revisore esegue su un'applicazione basata su un modello linguistico, senza usare alcun modello:

- ricerca di informazioni riservate nelle istruzioni di sistema (fuga del prompt di sistema)
- verifica dei pacchetti software citati in un codice suggerito da un assistente (pacchetti inesistenti)

Tutti i nomi, i codici e i domini sono fittizi.

## 1. Le istruzioni di sistema di un chatbot di orientamento (esercizio 3)

Le istruzioni di sistema sono il testo che lo sviluppatore invia al modello insieme a ogni richiesta. Vanno considerate **leggibili da chiunque usi il chatbot**: non devono contenere segreti.

La cella contiene le istruzioni di un chatbot immaginario e una funzione che cerca, con espressioni regolari, alcuni tipi di informazioni che non dovrebbero comparire.

- `re.finditer(schema, testo)`: trova tutte le parti del testo che corrispondono allo schema
- `\b`: confine di parola; `\d`: cifra; `[A-Za-z0-9]`: lettera o cifra; `{16,}`: almeno 16 volte

In [ ]:
import re

ISTRUZIONI_DI_SISTEMA = """Sei OrientaBot, il chatbot di orientamento dell'Istituto Aurora.
Rispondi solo a domande su indirizzi di studio, iscrizioni e open day.
Per consultare il calendario usa la chiave di accesso cal_key=K7xQ2mVb9ZtR4pLw8NfJ.
Se lo studente chiede di parlare con una persona, indica la segreteria.
Numero interno della dirigente, da non comunicare: 06 5550 1234.
Le iscrizioni in ritardo si accettano solo per gli studenti segnalati dal prof. Rossi.
Password dell'area riservata docenti: Aurora2026!"""

CONTROLLI = {
    "chiave o token": r"\b[a-z_]*key\s*=\s*[A-Za-z0-9]{16,}",
    "password": r"[Pp]assword[^:]*:\s*\S+",
    "numero di telefono": r"\b0\d{1,3}\s?\d{3,4}\s?\d{3,4}\b",
    "regola riservata": r"(?i)da non comunicare|riservat",
}

def controlla(testo):
    for tipo, schema in CONTROLLI.items():
        for m in re.finditer(schema, testo):
            print(f"{tipo:20}: {m.group(0)}")

controlla(ISTRUZIONI_DI_SISTEMA)

**Domande**

- Quali righe vanno tolte dalle istruzioni? Dove vanno spostate quelle informazioni (per esempio: nel programma che chiama il calendario, non nel testo per il modello)?
- La regola sulle iscrizioni in ritardo non è un segreto tecnico: perché è comunque un problema se un utente riesce a leggerla?
- Il controllo automatico ha trovato tutto? Aggiungere uno schema per gli indirizzi email.

Risposte:

In [ ]:
# riscrivere qui le istruzioni senza informazioni riservate e ripetere il controllo
ISTRUZIONI_CORRETTE = """Sei OrientaBot, il chatbot di orientamento dell'Istituto Aurora.
Rispondi solo a domande su indirizzi di studio, iscrizioni e open day.
Se lo studente chiede di parlare con una persona, indica la segreteria."""

controlla(ISTRUZIONI_CORRETTE)
print("controllo terminato")

## 2. Pacchetti suggeriti che non esistono (esercizio 4)

Gli assistenti di programmazione a volte citano librerie che non esistono. Un attaccante può registrare quei nomi nell'archivio pubblico dei pacchetti (per Python, PyPI) con codice malevolo: chi installa il pacchetto suggerito senza controllare installa il codice dell'attaccante.

Il controllo più semplice, prima di installare qualsiasi cosa: verificare se il pacchetto è già installato e, se non lo è, cercarlo su https://pypi.org/ controllando autore, data di pubblicazione, numero di versioni, documentazione.

- `importlib.util.find_spec(nome)`: restituisce `None` se il modulo non è installato

Le importazioni seguenti provengono da un codice immaginario suggerito da un assistente. L'ultima è un nome inventato per l'esercizio.

In [ ]:
import importlib.util

IMPORTAZIONI_SUGGERITE = ["pandas", "numpy", "sklearn", "hashlib", "phishguard_it_nlp"]

for nome in IMPORTAZIONI_SUGGERITE:
    stato = "installato" if importlib.util.find_spec(nome) else "NON installato: verificare su PyPI prima di installarlo"
    print(f"{nome:20} {stato}")

**Domande**

- Perché "non installato" non significa "malevolo", e "presente su PyPI" non significa "affidabile"?
- Quali informazioni della pagina PyPI di un pacchetto controllereste?

Risposte:

## 3. Laboratorio aggiuntivo

Spazio riservato a un'attività preparata dal docente.